# GTEx Skeletal-Muscle Preprocessing

This notebook runs preprocessing stage 1, reviews the saved expression matrix, then runs finalization and verification. Exploratory network, module, enrichment, and app work is in `bio_int.ipynb`.

## Notebook setup

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

%matplotlib inline

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "pipeline").is_dir() and (candidate / "data").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the Gene Co-Expression project root")

SRC_ROOT = project_root / "src"
PROCESSED = SRC_ROOT / "processed"
CONFIG = SRC_ROOT / "config"
DATA = project_root / "data"


def plot_pca_by_metadata(columns, labels=None, *, figsize=(8, 6)):
    labels = labels or {}
    for col in columns:
        plt.figure(figsize=figsize)
        points = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=metadata[col])
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.title(f"PCA colored by {labels.get(col, col)}")
        plt.colorbar(points, label=col)
        plt.show()


## Run preprocessing stage 1

In [ ]:
stage1 = subprocess.run(
    [sys.executable, "-m", "pipeline.preprocessing"],
    cwd=SRC_ROOT, check=True, capture_output=True, text=True,
)
print(stage1.stdout)

log_cpm = pd.read_parquet(PROCESSED / "muscle_logcpm_filtered.parquet")
metadata = pd.read_parquet(PROCESSED / "muscle_metadata_clean.parquet")
symbols = pd.read_parquet(PROCESSED / "gene_symbols.parquet")["symbol"]
print("Expression:", log_cpm.shape, "| metadata:", metadata.shape)

## Check expression distributions

### Global distribution

In [ ]:
plt.hist(log_cpm.to_numpy().flatten(), bins=100)
plt.xlabel("log-CPM"); plt.ylabel("Frequency")
plt.title("Distribution of Gene Expression")
plt.show()

### Per-sample distributions

In [ ]:
# Look for samples with shifted or unusually tall boxes
plt.figure(figsize=(24, 9))
log_cpm.boxplot(showfliers=False)
plt.ylabel("log-CPM"); plt.title("Expression Distribution Across Samples")
plt.xticks([])
plt.show()

### Sample-level summaries

In [ ]:
sample_means = log_cpm.mean(axis=0)
print(sample_means.describe())
print(log_cpm.median(axis=0).describe())
sample_means.plot(kind="hist", bins=30)
plt.xlabel("Mean log-CPM"); plt.ylabel("Number of samples")
plt.title("Distribution of Sample Mean Expression")
plt.show()

## Check sample correlations

In [ ]:
sample_corr = log_cpm.corr(method="pearson")
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(sample_corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(image, ax=ax, label="Pearson r")
ax.set(title="Sample-sample correlation", xlabel="Samples", ylabel="Samples")
plt.show()
# Each sample's average correlation with all others (lowest = most atypical)
mean_corr = (sample_corr.sum(axis=1) - 1) / (len(sample_corr) - 1)
print(mean_corr.sort_values().head(10))


### Observation: Sample similarity

The 803 samples have high average pairwise correlations. This plot is used for quality review; samples are not removed by a correlation cutoff.

## Inspect principal components

### Fit principal components

In [ ]:
# Samples are rows; use the same 10-component randomized PCA display as finalization.
X = log_cpm.T.to_numpy(dtype=np.float64, copy=False)
pca = PCA(n_components=10, svd_solver="randomized", random_state=42)
X_pca = pca.fit_transform(X)
explained = pca.explained_variance_ratio_
print("PC1-3:", explained[:3].round(3), "| first 10 cumulative:", round(explained.sum(), 3))


### Observation: PCA

PC1 and PC2 show broad sample structure, but the plot alone does not identify its cause. The next checks compare measured sample factors with these PCs.

### Plot principal components

The first 10 PCs are used for QC; PC1–PC3 explain about 22%, 14%, and 5% of variance. The finalizer plots 10 components and checks the first 3 against the configured covariates. Seed 42 makes randomized PCA repeatable.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1])
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("PCA of GTEx Muscle Samples")
plt.show()
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), explained[:10], marker="o")
plt.xlabel("Principal Component"); plt.ylabel("Proportion of Variance Explained")
plt.title("PCA Variance Explained")
plt.show()

## Review technical covariates and batch effects

### Align metadata to samples

In [ ]:
assert metadata.index.equals(log_cpm.columns)

### Add donor sex

GTEx codes sex as 1 = male and 2 = female. The notebook recodes this as 1 = male and 0 = female.

In [ ]:
pheno = pd.read_csv(DATA / "GTEx_Analysis_v8_Annotations_SubjectPhenotypesDS.txt", sep="\t")
donor_ids = pd.Series(log_cpm.columns).str.extract(r"^(GTEX-[^-]+)")[0].values
sex_raw = pheno.set_index("SUBJID")["SEX"].reindex(donor_ids).values
assert not np.isnan(sex_raw).any(), "Some samples have no sex annotation"
metadata["SEX"] = (sex_raw == 1).astype(int)       # 1 = male, 0 = female
print(metadata["SEX"].value_counts().rename({1: "male", 0: "female"}))

### Review selected technical variables

In [ ]:
manual_vars = {
    "SMRIN": "RNA Integrity (RIN)",
    "SMESTLBS": "Estimated Library Size",
    "SMMAPRT": "Mapping Rate",
    "SMDPMPRT": "Duplication Rate",
}
plot_pca_by_metadata(manual_vars, labels=manual_vars)

### Screen numeric attributes against PC1 and PC2

In [ ]:
numeric_metadata = metadata.select_dtypes(include="number")
numeric_metadata = numeric_metadata.loc[:, numeric_metadata.nunique(dropna=True) > 1]
pc1 = pd.Series(X_pca[:, 0], index=log_cpm.columns)
pc2 = pd.Series(X_pca[:, 1], index=log_cpm.columns)
pca_qc = pd.DataFrame({
    "PC1": numeric_metadata.corrwith(pc1),
    "PC2": numeric_metadata.corrwith(pc2),
})
pca_qc["max_abs_corr"] = pca_qc[["PC1", "PC2"]].abs().max(axis=1)
pca_qc = pca_qc.sort_values("max_abs_corr", ascending=False)
print(pca_qc.head(10))

### Plot strongest numeric associations

In [ ]:
top_attributes = pca_qc.head(4).index.tolist()  # strongest PC1/PC2 associations
plot_pca_by_metadata(top_attributes)

### Observation: Numeric factors

`SMNTRNRT` and `SMEXNCRT` track PC2 in opposite directions. They are strongly correlated with each other, so the final model fits them together.

### Check sex as a possible confounder

Y-linked genes should be expressed mainly in male samples. This check also shows whether sex aligns with the first two PCs.

In [ ]:
def log_cpm_row(symbol):
    genes = symbols.index[symbols.eq(symbol).fillna(False)]
    genes = genes.intersection(log_cpm.index)
    return log_cpm.loc[genes[0]] if len(genes) else None

ya, yb = log_cpm_row("RPS4Y1"), log_cpm_row("EIF1AY")
colors = np.where(metadata["SEX"] == 1, "tab:blue", "tab:orange")
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
if ya is not None and yb is not None:
    axes[0].scatter(ya, yb, c=colors, s=8, alpha=0.6)
    axes[0].set_xlabel("RPS4Y1"); axes[0].set_ylabel("EIF1AY")
    axes[0].set_title("Y-linked genes (blue = male, orange = female)")
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, s=8, alpha=0.6)
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2"); axes[1].set_title("PCA colored by sex")
plt.tight_layout(); plt.show()
print("SEX correlation with PC1/PC2:", pca_qc.loc["SEX", ["PC1", "PC2"]].round(3).to_dict())

### Observation: Sex

The Y-linked genes separate by sex, although sex correlates weakly with PC1 and PC2. The final correction includes sex to avoid gene correlations driven by this difference.

### Screen categorical batch variables

Adjusted R² estimates how much of PC1–PC3 is associated with each categorical factor. This helps avoid favoring factors just because they have many categories.

In [ ]:
cat_cols = [c for c in metadata.columns
            if not pd.api.types.is_numeric_dtype(metadata[c]) and 2 <= metadata[c].nunique() <= 100]
def adj_r2(pc, groups):
    groups = pd.Series(groups, index=pc.index)
    n, k = len(pc), groups.nunique()
    ss_tot = ((pc - pc.mean()) ** 2).sum()
    ss_bet = sum(len(g) * (g.mean() - pc.mean()) ** 2 for _, g in pc.groupby(groups))
    r2 = ss_bet / ss_tot
    return 1 - (1 - r2) * (n - 1) / (n - k) if n > k else np.nan
pcs = {f"PC{i+1}": pd.Series(X_pca[:, i], index=log_cpm.columns) for i in range(3)}
batch_qc = pd.DataFrame({name: {c: adj_r2(pc, metadata[c]) for c in cat_cols}
                         for name, pc in pcs.items()})
batch_qc["levels"] = [metadata[c].nunique() for c in batch_qc.index]
batch_qc["max_adj_r2"] = batch_qc[["PC1", "PC2", "PC3"]].max(axis=1)
batch_qc = batch_qc.sort_values("max_adj_r2", ascending=False)
display(batch_qc.round(3))
# PC1 vs PC2 for the top batch variables (only those with few levels)
for col in batch_qc.index[:2]:
    if metadata[col].nunique() <= 12:
        plt.figure(figsize=(7, 5))
        for lvl, idx in metadata.groupby(col).groups.items():
            plt.scatter(pcs["PC1"][idx], pcs["PC2"][idx], s=8, alpha=0.6, label=str(lvl))
        plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title(f"PCA colored by {col}")
        plt.legend(markerscale=2, fontsize=8)
        plt.show()

### Observation: Batch structure

Collection center has the largest association, at about 9% of PC variance; sequencing batch type is below 1%. The current preprocessing config does not add batch correction.

> `SMNTRNRT` and `SMEXNCRT` have strong, opposite correlations with PC2 and are fit together. Sex is also included because of its effect on Y-linked genes; the final model additionally includes `DTHHRDY`.

## Review sample-correlation QC

Use the correlation summary to inspect sample similarity. The current GTEx sample set is retained without an automatic correlation-based removal rule.

In [ ]:
print("Samples reviewed:", len(mean_corr))
print("Mean-correlation range:", round(mean_corr.min(), 3), "to", round(mean_corr.max(), 3))
print("No samples are removed by correlation in this pipeline.")

## Covariate decisions

Omitting `DTHHRDY` changed network hubs and connectivity, so it was added as a categorical covariate with missing values kept as a level. A separate `SME2MPRT` sensitivity check left the main module structure similar, so it remains outside the locked correction.

### Read the settings used by the preprocessing code

In [ ]:
preprocessing_config = json.loads((CONFIG / "preprocessing_params.json").read_text())
print(json.dumps(preprocessing_config, indent=2))


The config records skeletal-muscle RNA-seq selection, CPM filtering, covariate adjustment, and selection of the 5,000 highest-variance genes. PCA settings are for diagnostics.

## Run final preprocessing and EDA reports

In [ ]:
finalization = subprocess.run(
    [sys.executable, "-m", "pipeline.finalize_preprocessing"],
    cwd=SRC_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(finalization.stdout)


## Verify the saved preprocessing outputs

In [ ]:
verification = subprocess.run(
    [sys.executable, "-m", "verification.verify_preprocessing"],
    cwd=SRC_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(verification.stdout)


The finalizer writes corrected expression, metadata, symbols, parameter and EDA summaries, and six QC figures under `src/processed/`. The verifier checks dimensions, identifiers, alignment, and figure files.